# PHASE 2 — LLM định giá lại vùng mờ

Kernel **mới** (CUDA sạch, không torch-encoder tranh chấp → không lỗi init).
Đọc `phase1/` + `todo_llm/`, dùng Qwen quyết vùng mờ với **constrained decoding**
+ **ngưỡng cao** để chống ảo giác và tránh trích thừa.

**Chống ảo giác Qwen (bằng CODE, không tin lời model):**
1. `choice=[...]` — output chỉ **một chữ cái**, không thể bịa text.
2. Ứng viên **cắt từ văn bản gốc** — model không được gõ text.
3. Đọc **logprob → hậu nghiệm P**; **P < ngưỡng ⇒ BỎ** (thà sót).
4. Mỗi câu hỏi luôn có lựa chọn **"không chắc / KHÔNG_PHẢI"** để model không bị ép chọn.

**Metric:** sai type bị tính 2 lần đều 0 điểm ⇒ span mà LLM lưỡng lự type ⇒ **BỎ hẳn span**.

**Cần add Dataset chứa `phase1/` + `todo_llm/` (output của Phase 1).** GPU · Internet ON

In [ ]:
# Cell 1 — env spawn (vLLM là thứ DUY NHẤT chạm CUDA) + cài vllm
import os
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD','spawn')
import sys, glob, json, math, subprocess, time
subprocess.run([sys.executable,'-m','pip','install','-q','vllm'])
print('vllm cài xong')

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
# Cell 2 — dò phase1/todo + nạp Qwen 1 lần
def find_dir(name):
    h = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(x)]
    return h[0] if h else (name if os.path.isdir(name) else None)
P1  = find_dir('phase1');  TODO = find_dir('todo_llm')
assert P1 and TODO, 'thiếu phase1/ hoặc todo_llm/ — add Dataset output của Phase 1'
if not os.path.exists('fakeer'): subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'])
sys.path.insert(0, 'fakeer/src')
os.makedirs('/kaggle/working/final', exist_ok=True)

LLM_MODEL = 'Qwen/Qwen3-8B'   # ≤9B; OOM -> Qwen/Qwen2.5-7B-Instruct
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from transformers import AutoTokenizer
qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
# Qwen3-8B fp16 = 16.4GB > 14.56GB CỦA MỘT T4 -> OOM. Dùng CẢ HAI GPU
# (tensor_parallel_size=2, mỗi con ~8.2GB) và hạ util để chừa chỗ KV-cache.
# max_model_len 2048 là đủ: prompt chỉ có ngữ cảnh ngắn + 1 cụm, sinh 4 token.
import torch as _t
_ngpu = max(1, _t.cuda.device_count())
print(f'{_ngpu} GPU')

def _mk(model, tp):
    return LLM(model=model, dtype='float16', max_model_len=2048,
               gpu_memory_utilization=0.88, tensor_parallel_size=tp,
               enforce_eager=True, disable_custom_all_reduce=True,
               swap_space=2, enable_prefix_caching=True)

# Chỉ 1 GPU T4 (14.56GB) thì Qwen3-8B fp16 (16.4GB) KHÔNG vừa -> lùi 7B.
# Không để OOM giết cả notebook sau khi Phase 1 đã chạy xong.
try:
    llm = _mk(LLM_MODEL, _ngpu)
except Exception as _e:
    print('⚠️', type(_e).__name__, str(_e)[:100])
    LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
    print('-> lùi sang', LLM_MODEL)
    qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
    llm = _mk(LLM_MODEL, _ngpu)
print('Qwen sẵn sàng')

# path + import SỚM: bản cũ import TRƯỚC sys.path.insert -> ModuleNotFoundError
import glob as _g
_src = next(iter(_g.glob('/kaggle/working/fakeer/src') + _g.glob('/kaggle/input/**/src', recursive=True)), None)
if _src is None:
    subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer']); _src='fakeer/src'
sys.path.insert(0, _src)
from utils.overlap_resolver import select_non_overlapping
print('overlap_resolver ok từ', _src)

In [ ]:
# Cell 3 — hàm hỏi Qwen: constrained choice + hậu nghiệm + NGƯỠNG
TAU_TYPE = 0.75   # span mờ: P < ngưỡng -> BỎ span (sai type phạt kép)
TAU_LAB  = 0.80   # ứng viên xét nghiệm: P < ngưỡng -> BỎ (trích thừa phạt x3)

def _posterior(o, keys):
    lp = (getattr(o, 'logprobs', None) or [None])[0]
    if lp:
        sc = {}
        for _t, x in lp.items():
            t = (getattr(x,'decoded_token',None) or '').strip()
            if t in keys: sc[t] = max(sc.get(t,-1e9), x.logprob)
        if sc:
            m = max(sc.values()); ex = {k: math.exp(v-m) for k,v in sc.items()}; Z = sum(ex.values())
            b = max(ex, key=ex.get); return b, ex[b]/Z
    g = (getattr(o,'text','') or '').strip()[:1]
    return (g if g in keys else keys[-1]), 0.0

def ask_batch(system, tails, keys):
    sp = SamplingParams(temperature=0, max_tokens=4, logprobs=20,
                        structured_outputs=StructuredOutputsParams(choice=keys))
    prompts = [qtok.apply_chat_template(
        [{'role':'system','content':system},{'role':'user','content':t}],
        tokenize=False, add_generation_prompt=True) for t in tails]
    return [_posterior(r.outputs[0], keys) for r in llm.generate(prompts, sp)]

SYS_TYPE = ("Bạn là bác sĩ. Với cụm từ trích NGUYÊN VĂN từ bệnh án, chọn đúng một chữ cái:\n"
    "A. TRIỆU_CHỨNG — biểu hiện/dấu hiệu bệnh nhân khai hoặc bác sĩ quan sát (nặng mặt, tiểu ít, khó thở).\n"
    "B. CHẨN_ĐOÁN — tên một bệnh/hội chứng được kết luận (Viêm cầu thận mạn, Hội chứng thận hư, sỏi mật).\n"
    "C. KHÔNG_RÕ — không đủ chắc là A hay B.\n"
    "Chỉ trả một chữ cái. Nếu phân vân, chọn C.")

SYS_LAB = ("Bạn là bác sĩ. Với cụm từ trích NGUYÊN VĂN từ bệnh án, chọn đúng một chữ cái:\n"
    "A. TÊN_XÉT_NGHIỆM — tên xét nghiệm cận lâm sàng / chỉ số hoá sinh, huyết học (Ure, Creatinin, WBC, CRP).\n"
    "B. KẾT_QUẢ_XÉT_NGHIỆM — giá trị đo được của xét nghiệm cận lâm sàng (6,4 mmol/l, 92 g/l).\n"
    "C. KHÔNG_PHẢI — KHÔNG phải xét nghiệm cận lâm sàng: gồm dấu hiệu sinh tồn (huyết áp, mạch, "
    "nhiệt độ, nhịp thở, SpO2), tuổi, liều thuốc, hay số liệu thống kê.\n"
    "Chỉ trả một chữ cái. Nếu không chắc là xét nghiệm, chọn C.")
print('prompt sẵn sàng | TAU_TYPE', TAU_TYPE, '| TAU_LAB', TAU_LAB)

In [ ]:
# Cell 4 — duyệt từng file: định giá vùng mờ, chỉ GIỮ khi vượt ngưỡng
from typing import List
def ctx(text, s, e, w=80):
    return text[max(0,s-w):min(len(text),e+w)].replace('\n',' ')

ids = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{P1}/*.json'))
n_add_sym = n_add_lab = n_drop = 0
for fid in ids:
    res  = json.load(open(f'{P1}/{fid}.json', encoding='utf-8'))
    todo = json.load(open(f'{TODO}/{fid}.json', encoding='utf-8'))
    TEXT = res['text']; MAP = {'A':'TRIỆU_CHỨNG','B':'CHẨN_ĐOÁN'}; MAPL = {'A':'TÊN_XÉT_NGHIỆM','B':'KẾT_QUẢ_XÉT_NGHIỆM'}
    kb_ents = list(res['entities'])          # KB+luật: ƯU TIÊN TUYỆT ĐỐI
    kb_iv = [(e['start'], e['end']) for e in kb_ents]
    llm_new = []                              # LLM chỉ được thêm vào CHỖ TRỐNG

    # (1) span SYM_DIS chưa quyết type: A/B/C ; C hoặc P thấp -> BỎ span
    su = todo.get('sym_undecided', [])
    if su:
        tails = [f"Đoạn: «{ctx(TEXT,s['start'],s['end'])}»\n\nCụm: «{s['text']}»\n\nNhãn:" for s in su]
        for s,(k,p) in zip(su, ask_batch(SYS_TYPE, tails, ['A','B','C'])):
            if k in MAP and p >= TAU_TYPE:
                llm_new.append({'text':s['text'],'type':MAP[k],'start':s['start'],'end':s['end'],
                            'score':round(p,3),'source':'encoder+llm','negated':False,'assertion':'affirmed'})
                n_add_sym += 1
            else: n_drop += 1

    # (2) ứng viên xét nghiệm vùng mờ: A/B/C ; C hoặc P thấp -> BỎ
    lc = todo.get('lab_candidates', [])
    if lc:
        tails = [f"Đoạn: «{ctx(TEXT,c['start'],c['end'])}»\n\nCụm: «{c['text']}»\n\nNhãn:" for c in lc]
        for c,(k,p) in zip(lc, ask_batch(SYS_LAB, tails, ['A','B','C'])):
            if k in MAPL and p >= TAU_LAB:
                llm_new.append({'text':c['text'],'type':MAPL[k],'start':c['start'],'end':c['end'],
                            'score':round(p,3),'source':'llm','negated':False,'assertion':'affirmed'})
                n_add_lab += 1

    # KB THẮNG LLM: bỏ span LLM chồng lên span KB. Nếu để DP tự chọn theo score,
    # một span LLM DÀI có thể thắng span KB NGẮN dù KB chắc hơn -> đổi type ->
    # đề tính 2 lần, mỗi lần 0 điểm cả 3 metric. Chặn bằng cấu trúc, không bằng score.
    llm_new = [e for e in llm_new
               if not any(e['start'] < b and a < e['end'] for a, b in kb_iv)]
    for e in kb_ents: e.setdefault('score', 1.0)
    kept = select_non_overlapping(kb_ents + llm_new); kept.sort(key=lambda x:x['start'])
    assert all(e['text']==TEXT[e['start']:e['end']] for e in kept), fid
    o = sorted(kept, key=lambda x:x['start'])
    assert all(o[i]['end']<=o[i+1]['start'] for i in range(len(o)-1)), f'{fid} chồng lấn'
    json.dump({'text':TEXT,'entities':kept}, open(f'/kaggle/working/final/{fid}.json','w',encoding='utf-8'), ensure_ascii=False)
print(f'\nXONG | +{n_add_sym} chẩn/triệu (LLM) | +{n_add_lab} xét nghiệm (LLM) | bỏ {n_drop} span type không chắc')
print('=> /kaggle/working/final/*.json')